第 1 段：安装依赖库

In [1]:
# 安装所需库（在 Jupyter 中运行一次即可）
!pip install scikit-learn nltk textblob transformers torch

# 下载 NLTK 的 VADER 词典（后续使用）
import nltk
nltk.download('vader_lexicon')

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/625.0 kB ? eta -:--:

C:\Users\Administrator\.conda\envs\rl\lib\ssl.py:570: UserWarning: unable to load Windows certificates, some may be corrupted
  warnings.warn("unable to load Windows certificates, some may be corrupted")
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...


True

第 2 段：基于词典的情感分析（伪代码）

In [2]:
# 基于词典的情感分析（伪代码框架）
def lexicon_based_sentiment(text):
    sentiment_score = 0
    words = tokenize(text)  # 分词

    for word in words:
        if word in positive_lexicon:
            sentiment_score += positive_lexicon[word]
        elif word in negative_lexicon:
            sentiment_score -= negative_lexicon[word]

    # 处理否定和程度修饰
    sentiment_score = apply_negation(words, sentiment_score)
    sentiment_score = apply_intensifier(words, sentiment_score)

    return normalize(sentiment_score)

第 3 段：使用 VADER 进行情感分析（补充）

In [3]:
from nltk.sentiment import SentimentIntensityAnalyzer

# 初始化 VADER 分析器
sia = SentimentIntensityAnalyzer()

# 分析文本
texts = [
    "This product is absolutely amazing! I love it.",
    "Terrible service, very disappointed.",
    "The movie was okay, nothing special."
]

for text in texts:
    scores = sia.polarity_scores(text)
    print(f"文本: {text}")
    print(f"情感分数: {scores}")
    print(f"综合判断: {'正面' if scores['compound'] >= 0.05 else '负面' if scores['compound'] <= -0.05 else '中性'}\n")

文本: This product is absolutely amazing! I love it.
情感分数: {'neg': 0.0, 'neu': 0.361, 'pos': 0.639, 'compound': 0.8709}
综合判断: 正面

文本: Terrible service, very disappointed.
情感分数: {'neg': 0.765, 'neu': 0.235, 'pos': 0.0, 'compound': -0.7574}
综合判断: 负面

文本: The movie was okay, nothing special.
情感分数: {'neg': 0.277, 'neu': 0.49, 'pos': 0.233, 'compound': -0.092}
综合判断: 负面



第 4 段：使用 TextBlob 进行情感分析（补充）

In [5]:
from textblob import TextBlob

# 分析文本
texts = [
    "This product is absolutely amazing! I love it.",
    "Terrible service, very disappointed.",
    "The movie was okay, nothing special."
]

for text in texts:
    blob = TextBlob(text)
    print(f"文本: {text}")
    print(f"极性 (Polarity): {blob.sentiment.polarity}")  # -1 到 1，负值表示负面，正值表示正面
    print(f"主观性 (Subjectivity): {blob.sentiment.subjectivity}")  # 0 到 1
    print()

文本: This product is absolutely amazing! I love it.
极性 (Polarity): 0.625
主观性 (Subjectivity): 0.75

文本: Terrible service, very disappointed.
极性 (Polarity): -0.9875
主观性 (Subjectivity): 0.9875

文本: The movie was okay, nothing special.
极性 (Polarity): 0.4285714285714286
主观性 (Subjectivity): 0.5357142857142857



第 5 段：使用 Scikit-learn 构建情感分类器

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

# 构建分类管道
sentiment_clf = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LinearSVC())
])

# 准备训练数据（示例数据，实际使用时需替换为真实数据集）
train_texts = [
    "这个产品非常好用，强烈推荐！",
    "质量太差了，完全不值得买。",
    "服务态度很好，下次还会来。",
    "发货速度慢，包装也有破损。"
]
train_labels = ['positive', 'negative', 'positive', 'negative']

# 训练模型
sentiment_clf.fit(train_texts, train_labels)

# 预测新文本
predictions = sentiment_clf.predict(["这个产品非常好用，强烈推荐！"])
print(predictions)  # 输出: ['positive']

['positive']


第 6 段：使用 BERT 进行方面级情感分析

In [2]:
from pathlib import Path

import torch
from transformers import (
    DebertaV2Tokenizer,
    AutoModelForSequenceClassification
)


# ============================================================
# 1. 本地模型目录
# ============================================================

model_dir = Path(
    r"D:\11\NLP\data\absa-sentiment-model"
)


# ============================================================
# 2. 检查必要文件
# ============================================================

required_files = [
    "config.json",
    "model.safetensors",
    "spm.model",
    "tokenizer_config.json"
]


missing_files = []

for file_name in required_files:
    file_path = model_dir / file_name

    if not file_path.is_file():
        missing_files.append(file_name)


if missing_files:
    raise FileNotFoundError(
        "模型文件不完整，缺少：\n"
        + "\n".join(missing_files)
        + f"\n\n请把文件放入：{model_dir}"
    )


# 检查spm.model大小，防止下载成错误页面或指针文件
spm_file = model_dir / "spm.model"
spm_size = spm_file.stat().st_size


if spm_size < 1_000_000:
    raise RuntimeError(
        f"spm.model文件明显过小：{spm_size}字节\n"
        "正常文件约为2.46 MB，请重新下载spm.model。"
    )


print("本地文件检查成功！")
print(
    "spm.model大小：",
    f"{spm_size / 1024 / 1024:.2f} MB"
)


# ============================================================
# 3. 明确加载DeBERTa慢速分词器
#
# 不再使用AutoTokenizer默认的快速分词器，
# 避免NoneType.endsWith错误
# ============================================================

tokenizer = DebertaV2Tokenizer.from_pretrained(
    str(model_dir),
    local_files_only=True
)


# ============================================================
# 4. 加载已经微调好的情感分类模型
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    str(model_dir),
    local_files_only=True,
    use_safetensors=True
)


model.eval()

print("分词器加载成功！")
print("情感分类模型加载成功！")


# ============================================================
# 5. 输入评论和评价方面
# ============================================================

text = "餐厅的环境很棒，但是服务太慢了。"
aspect = "服务"


# 官方用法是：
# 第一个文本是完整评论
# text_pair是需要分析的方面
#
# 模型输入大致为：
# [CLS] 评论 [SEP] 方面 [SEP]
inputs = tokenizer(
    text,
    text_pair=aspect,
    return_tensors="pt",
    truncation=True,
    max_length=128
)


# ============================================================
# 6. 查看模型实际收到的内容
# ============================================================

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)


print("\n评论：")
print(text)

print("\n评价方面：")
print(aspect)

print("\n分词结果：")
print(tokens)

print("\ninput_ids形状：")
print(inputs["input_ids"].shape)


# ============================================================
# 7. 模型预测
# ============================================================

with torch.no_grad():
    outputs = model(**inputs)


logits = outputs.logits

probabilities = torch.softmax(
    logits,
    dim=-1
)

prediction_id = probabilities.argmax(
    dim=-1
).item()


# ============================================================
# 8. 获取标签
# ============================================================

label = model.config.id2label.get(
    prediction_id,
    str(prediction_id)
)


label_to_chinese = {
    "Negative": "负面",
    "Neutral": "中性",
    "Positive": "正面"
}


chinese_label = label_to_chinese.get(
    label,
    label
)


# ============================================================
# 9. 输出每一类概率
# ============================================================

print("\n模型输出logits：")
print(logits)

print("\n各类别概率：")

for class_id, probability in enumerate(
    probabilities[0]
):
    class_label = model.config.id2label.get(
        class_id,
        str(class_id)
    )

    chinese_class_label = label_to_chinese.get(
        class_label,
        class_label
    )

    print(
        f"{class_id} - "
        f"{chinese_class_label:<6}"
        f"{probability.item():.4f}"
    )


print("\n最终预测结果：")
print(chinese_label)

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


本地文件检查成功！
spm.model大小： 2.35 MB
分词器加载成功！
情感分类模型加载成功！

评论：
餐厅的环境很棒，但是服务太慢了。

评价方面：
服务

分词结果：
['[CLS]', '▁', '餐', '厅', '的', '环', '境', '很', '棒', ',', '但', '是', '服务', '太', '慢', '了', '。', '[SEP]', '▁', '服务', '[SEP]']

input_ids形状：
torch.Size([1, 21])

模型输出logits：
tensor([[ 4.2425, -2.3464, -1.7157]])

各类别概率：
0 - 负面    0.9961
1 - 中性    0.0014
2 - 正面    0.0026

最终预测结果：
负面


补充：完整的 Scikit-learn 情感分类流水线（含数据准备）

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# 加载数据（选择两个类别作为正面/负面示例）
# 注意：20newsgroups 不是情感数据集，此处仅作演示用途
categories = ['alt.atheism', 'soc.religion.christian']
newsgroups = fetch_20newsgroups(subset='all', categories=categories)

# 分割数据集
X_train, X_test, y_train, y_test = train_test_split(
    newsgroups.data, newsgroups.target, test_size=0.2, random_state=42
)

# 构建 Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000))
])

# 训练
pipeline.fit(X_train, y_train)

# 预测与评估
y_pred = pipeline.predict(X_test)
print(f"准确率: {accuracy_score(y_test, y_pred):.2f}")
print("\n分类报告:")
print(classification_report(y_test, y_pred, target_names=categories))

准确率: 0.97

分类报告:
                        precision    recall  f1-score   support

           alt.atheism       1.00      0.93      0.96       159
soc.religion.christian       0.95      1.00      0.97       201

              accuracy                           0.97       360
             macro avg       0.97      0.97      0.97       360
          weighted avg       0.97      0.97      0.97       360



补充：使用朴素贝叶斯进行情感分类

In [2]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# 构建朴素贝叶斯分类管道
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', MultinomialNB())
])

# 训练与预测（使用上面的数据）
nb_pipeline.fit(X_train, y_train)
y_pred_nb = nb_pipeline.predict(X_test)
print(f"朴素贝叶斯准确率: {accuracy_score(y_test, y_pred_nb):.2f}")

朴素贝叶斯准确率: 0.96
